In [1]:
# Install necessary libraries
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers rank_bm25
!pip install -q chromadb langchain langchain-community langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 8.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.0 MB/s eta 0:

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
%pip install langchain_chroma
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"System ready! Using device: {device}")

# 2. Setup Your Embeddings (The code you provided)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Load Your ChromaDB
# --- UPDATE THIS PATH ---
DB_PATH = "/content/drive/MyDrive/graduation_project/vector_store/chroma_db"

if os.path.exists(DB_PATH):
    vector_db = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
    print(f"Vector Store loaded. Collection contains {vector_db._collection.count()} chunks.")
else:
    print(f"WARNING: Path {DB_PATH} not found. Please check the path.")

# 4. Load Qwen 7B (4-bit Quantized)
model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Qwen 7B...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Qwen 7B loaded successfully!")

System ready! Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen 7B loaded successfully!


In [6]:
import json
import re
from sentence_transformers import util # We still need this for the math part (cos_sim)

# --- CONFIGURATION ---
THRESHOLD_HIGH = 0.85
VALID_GRADES = [0.0, 0.25, 0.5, 0.75, 1.0]

# --- Helper 1: Hybrid Similarity (Using your LangChain Embeddings) ---
def calculate_hybrid_similarity(text1, text2):
    # 1. Semantic Score (Meaning)
    # We use your existing 'embeddings' object
    emb1 = embeddings.embed_query(text1)
    emb2 = embeddings.embed_query(text2)

    # Convert to tensor for calculation
    tensor1 = torch.tensor(emb1)
    tensor2 = torch.tensor(emb2)
    semantic_score = util.cos_sim(tensor1, tensor2).item()

    # 2. Lexical Score (Word Overlap - Jaccard)
    set1 = set(text1.lower().split())
    set2 = set(text2.lower().split())
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    lexical_score = intersection / union if union > 0 else 0

    # 3. Weighted Average
    return round((0.7 * semantic_score) + (0.3 * lexical_score), 3)

# --- Helper 2: RAG Retrieval ---
def get_diverse_context(question):
    try:
        # 'mmr' ensures diversity in retrieved chunks
        results = vector_db.search(query=question, search_type="mmr", k=3)
        if not results:
            return "No context found."
        return "\n\n".join([doc.page_content for doc in results])
    except Exception as e:
        return f"Error retrieving context: {str(e)}"

# --- Main Logic ---
def grade_submission(question, student_ans, model_ans):

    # === TIER 1: FAST LANE ===
    direct_score = calculate_hybrid_similarity(student_ans, model_ans)

    if direct_score >= THRESHOLD_HIGH:
        return {
            "grade": 1.0,
            "feedback": "Perfect match with the model answer.",
            "method": "Fast_Tier_Direct_Match"
        }

    # === TIER 2: RAG LANE ===
    context_chunks = get_diverse_context(question)

    prompt = f"""
    You are a strict academic grading AI.
    The student's answer did NOT match the Model Answer directly.
    Decide the grade based on the **Reference Context**.

    ### DATA:
    - **Question**: {question}
    - **Student Answer**: {student_ans}
    - **Model Answer**: {model_ans}
    - **Reference Context**: {context_chunks}

    ### RUBRIC (Pick Closest):
    - 1.0: Correct concept, supported by Context.
    - 0.75: Mostly correct, misses minor details.
    - 0.5: Partially correct, significant gaps.
    - 0.25: Mostly wrong, mentions 1 correct keyword.
    - 0.0: Wrong.

    ### OUTPUT JSON ONLY:
    {{
        "grade": <number>,
        "feedback": "Two sentences max."
    }}
    """

    messages = [
        {"role": "system", "content": "Output only valid JSON."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=150,
        temperature=0.05,
        do_sample=True
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    try:
        json_match = re.search(r"\{.*\}", response, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group(0))
            raw_grade = float(result.get("grade", 0))
            final_grade = min(VALID_GRADES, key=lambda x: abs(x - raw_grade))

            return {
                "grade": final_grade,
                "feedback": result.get("feedback", "No feedback."),
                "method": "RAG_LLM_Check"
            }
        else:
            return {"grade": 0.0, "feedback": "AI Format Error", "raw": response}
    except Exception as e:
        return {"grade": 0.0, "feedback": f"System Error: {str(e)}"}

In [7]:
# Test your system with hardcoded examples
test_cases = [
    {
        "type": "Exact Match",
        "Question": "What is CPU?",
        "Model_Answer": "Central Processing Unit",
        "Student_Answer": "Central Processing Unit"
    },
    {
        "type": "Correct but RAG needed",
        "Question": "Explain the stack data structure.",
        "Model_Answer": "Stack follows LIFO (Last In First Out) principle.",
        "Student_Answer": "It is a linear structure where insertion and deletion happen at one end, like a pile of plates."
    },
    {
        "type": "Wrong Answer",
        "Question": "What is Python?",
        "Model_Answer": "Python is a high-level programming language.",
        "Student_Answer": "Python is a type of snake in the jungle."
    }
]

print(f"{'TEST TYPE':<25} | {'GRADE':<6} | {'METHOD':<20} | FEEDBACK")
print("-" * 100)

for case in test_cases:
    output = grade_submission(case["Question"], case["Student_Answer"], case["Model_Answer"])
    print(f"{case['type']:<25} | {output['grade']:<6} | {output['method']:<20} | {output['feedback']}")

TEST TYPE                 | GRADE  | METHOD               | FEEDBACK
----------------------------------------------------------------------------------------------------
Exact Match               | 1.0    | Fast_Tier_Direct_Match | Perfect match with the model answer.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Correct but RAG needed    | 0.75   | RAG_LLM_Check        | The student correctly identified that insertion and deletion happen at one end, but missed the key LIFO principle.
Wrong Answer              | 0.0    | RAG_LLM_Check        | The student's answer is incorrect and does not match the question.


In [10]:
# IF INPUT IS JSON
import json
from google.colab import files
from tqdm import tqdm

# ==========================================
# 1. UPLOAD JSON FILE
# ==========================================
print("Please upload your JSON file containing the student data...")
uploaded = files.upload()

# Get the filename (assuming only one file is uploaded)
filename = next(iter(uploaded))
print(f"Processing file: {filename}...")

# Load the content
with open(filename, 'r', encoding='utf-8') as f:
    student_data = json.load(f)

# ==========================================
# 2. BATCH GRADING PROCESS
# ==========================================
print(f"Starting Batch Grading for {len(student_data)} questions...")

final_results = []

# Iterate through the data with a progress bar
for entry in tqdm(student_data):
    try:
        # Extract necessary fields
        # Using .get() ensures it doesn't crash if a key is missing, but assumes standard structure
        q = entry.get('Question', "")
        s = entry.get('Student_answer', "")
        m = entry.get('Model_answer', "")

        # Run the Grading Function
        grading_result = grade_submission(q, s, m)

        # Create a complete record (Preserve ID/Name + Add AI Results)
        processed_entry = entry.copy()

        # Add the new fields requested
        processed_entry['grade'] = grading_result['grade']
        processed_entry['feedback'] = grading_result['feedback']
        processed_entry['Method_Used'] = grading_result['method']

        final_results.append(processed_entry)

    except Exception as e:
        # Error handling to keep the process running
        error_entry = entry.copy()
        error_entry['grade'] = 0.0
        error_entry['feedback'] = f"System Error: {str(e)}"
        error_entry['Method_Used'] = "Error"
        final_results.append(error_entry)

# ==========================================
# 3. SAVE AND DOWNLOAD RESULTS
# ==========================================
output_filename = "graded_output.json"

# Save as JSON
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print(f"\nGrading Complete! Saving {output_filename}...")
files.download(output_filename)

# Preview the first entry to confirm structure
if final_results:
    print("\n--- Preview of First Graded Item ---")
    print(json.dumps(final_results[0], indent=2))

Please upload your JSON file containing the student data...


Saving test_kareem.json to test_kareem (1).json
Processing file: test_kareem (1).json...
Starting Batch Grading for 27 questions...


100%|██████████| 27/27 [00:25<00:00,  1.07it/s]


Grading Complete! Saving graded_output.json...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Preview of First Graded Item ---
{
  "Student_ID": 211000606,
  "Name": "Karim Fawzy.",
  "Question": "Q1.1: Choose the suitable answer",
  "Student_answer": "b",
  "Model_answer": "b",
  "grade": 1.0,
  "feedback": "Perfect match with the model answer.",
  "Method_Used": "Fast_Tier_Direct_Match"
}


In [8]:
import pandas as pd
from google.colab import files
from tqdm import tqdm

# 1. Upload File
print("Upload your 'students.csv' file (Must have columns: question, student_answer, model_answer)")
filename = "/content/drive/MyDrive/graduation_project/data_raw/data_khawaga_4_10.csv"

# 2. Process
df = pd.read_csv(filename)
results = []

print("Starting Batch Grading...")
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    try:
        q = row['question']
        s = row['student_answer']
        m = row['model_answer']

        # Run Grading
        res = grade_submission(q, s, m)

        # Save info
        results.append({
            "AI_Grade": res['grade'],
            "AI_Feedback": res['feedback'],
            "Method_Used": res['method']
        })
    except Exception as e:
        results.append({"AI_Grade": 0, "AI_Feedback": "Error", "Method_Used": "Fail"})



Upload your 'students.csv' file (Must have columns: question, student_answer, model_answer)
Starting Batch Grading...


  0%|          | 13/8352 [01:05<11:35:05,  5.00s/it]


KeyboardInterrupt: 

In [ ]:
# 3. Save Results
results_df = pd.DataFrame(results)
final_df = pd.concat([df, results_df], axis=1)

output_file = "graded_results.csv"
final_df.to_csv(output_file, index=False)
print(f"Done! Saving {output_file}...")
files.download(output_file)

In [ ]:
print(len(results_df),'\n')
print(results_df.head())